# Allocation and Welfare Across Personal-Time Prices

Reproduce the finite five-condition, six-price allocation diagnostics and fixed true-state payoffs. The public policy producer supplies 450 native roots at each price. This analysis uses their checked terminal allocations and the original target-specific latent keys. It does not rerun decision-making or treat expected-only stops as historical observed trajectories.

Use Python 3.10, NumPy 1.23.5 and Node.js 22.14.0 with the [reference environment](../../configs/native_reference_environment.yml). Full mode is expensive; smoke checks interfaces only.

In [ ]:
from pathlib import Path
import subprocess, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-c', str(ROOT / 'configs/native_reference_constraints.txt'), '-e', str(ROOT)])

## Measurements

For each ten-trial schedule, regress the higher-initial-mean-need recipient's final share on the absolute initial mean difference divided by initial tutoring budget. Include an intercept, then equally average nine schedule coefficients. Initial ties are excluded from this rank-oriented regression; C5 is undefined, not zero. This differs from joint A/B amount slopes used in the statistical analysis.

The four distances compare the selected share with equal split, equal mean surplus, the saved feasible-grid expected-utility maximizer and all to the lower terminal mean need. They use the same terminal beliefs and budget. Allocation distance is not welfare regret; tied optima may be far apart. Expected and realized utility are the sum of recipient utilities, without subtracting personal-time costs.

Every realized value is a target-key fixed model evaluation. The historical study combined original outcomes, same-target reuse, 60 C5 fixed-outcome compatibility checks and 1,224 newly materialized target outcomes. Fresh reproduction does not recreate that archival history or claim equal historical/current probability vectors; it reconstructs the target-key outcomes. No population precision or human-power conclusion follows.

In [ ]:
import json, shlex
RUN = False
FULL = False
RUN_POLICY = False  # Enable to generate inputs; leave false to reuse a checked matching run.
NODE = 'node'
POLICY = ROOT / 'results' / ('personal_time_sensitivity_full' if FULL else 'personal_time_sensitivity_selected')
OUTPUT = ROOT / 'results' / ('allocation_welfare_full' if FULL else 'allocation_welfare_selected')
scope = ['--full'] if FULL else ['--only', 'C3:0:1,C5:0:1']
print(json.dumps(json.loads((ROOT / 'configs/personal_time_diagnostics.json').read_text()), indent=2))

In [ ]:
for mode in ('prepare', 'produce', 'check'):
    command = [sys.executable, str(ROOT / 'scripts/reproduce_personal_time_sensitivity.py'), mode, '--output', str(POLICY)] + scope
    if mode == 'prepare':
        command += ['--node', NODE]
    print(shlex.join(command))
    if RUN and RUN_POLICY:
        subprocess.check_call(command, cwd=ROOT)

In [ ]:
command = [sys.executable, str(ROOT / 'scripts/analyze_personal_time_allocations.py'), '--input', str(POLICY), '--output', str(OUTPUT)]
if FULL:
    command += ['--figures']
print(shlex.join(command))
if RUN:
    subprocess.check_call(command, cwd=ROOT)

## Outputs and Scope

`trial_metrics.json` retains every target. `summary.json` reports completeness and equal-schedule summaries; full mode supplies 30 cells, 24 defined rank coefficients, six undefined coefficients and 120 distances. `allocation_welfare.csv` contains complete cells only. `independent_check.json` records checked input/source identities and separate scalar/least-squares agreement. Partial runs never become full-roster evidence.

`initial_need_amounts.csv` exports logical-A allocated minutes and both initial means, with scientific trial identity and price. A joint A/B amount regression is a different estimand from the rank-oriented share regression computed here. This notebook does not fit that statistical model.

Full plots: `welfare_and_search.png`, `allocation_slopes.png`, `benchmark_distances.png`. Historical administrative provenance plots and report layout are not regenerated. Saved output files are not bundled in Git.

In [ ]:
if RUN:
    summary = json.loads((OUTPUT / 'summary.json').read_text())
    print({key: summary[key] for key in ('verifiedTrials', 'fullFiniteTarget', 'partialPersons')})
    if (OUTPUT / 'allocation_welfare.csv').exists():
        print((OUTPUT / 'allocation_welfare.csv').read_text())
    if FULL:
        from IPython.display import Image, display
        for name in ('welfare_and_search.png', 'allocation_slopes.png', 'benchmark_distances.png'):
            display(Image(filename=str(OUTPUT / name)))